In [8]:
from youtube_transcript_api import YouTubeTranscriptApi

# The video ID is the part of the URL after "v="
# Example: For https://www.youtube.com/watch?v=dQw4w9WgXcQ, the ID is "dQw4w9WgXcQ"
video_id = "rV3HJ4LEZ7k"
ytt_api = YouTubeTranscriptApi()

    # Fetch the default (usually English) transcript
transcript = ytt_api.fetch(video_id=video_id)
    


In [9]:
# Assuming your FetchedTranscript object is stored in a variable called 'transcript_data'

for snippet in transcript.snippets:
    # Get the start time and text from the object properties
    start_time = snippet.start
    
    # Replace newlines with spaces so the text stays on one line
    text = snippet.text.replace('\n', ' ') 
    
    # Calculate minutes and seconds
    minutes = int(start_time // 60)
    seconds = int(start_time % 60)
    
    # Print in [MM:SS] format
    print(f"[{minutes:02d}:{seconds:02d}] {text}")

[00:00] Hello all, my name is Krishna and
[00:02] welcome to my YouTube channel. So guys,
[00:04] super excited to bring this specific
[00:07] video which is more than 10.5
[00:10] hours and the best part about this video
[00:13] will be that from past four to 5 months
[00:16] every important topics that has actually
[00:19] evolved in AI specifically in the field
[00:22] of generative AI and agentic AI have
[00:24] covered almost everything. Just let me
[00:26] talk about the plan how we are going to
[00:28] cover. First of all, we are going to
[00:30] understand about generative AI and
[00:32] agentic AI with Langchain. Then we are
[00:35] going to see a langraph complete
[00:37] langraph crash course wherein we will
[00:39] focus on building agentic AI
[00:41] application. Then the third important
[00:44] part will be the entire rag you know how
[00:46] you can go ahead and implement rag and
[00:49] this will not be only traditional rag
[00:50] we'll also try to cover agentic rag an

In [11]:
import yt_dlp

video_url = "https://www.youtube.com/watch?v=rV3HJ4LEZ7k"

# Configure yt-dlp to only fetch metadata, not the video itself
ydl_opts = {
    "quiet": True,
    "skip_download": True, 
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    # extract_info fetches the metadata dictionary
    info = ydl.extract_info(video_url, download=False)
    print(info)
    
    # Safely get the chapters array (returns None if no chapters exist)
    chapters = info.get("chapters")
    
    if chapters:
        print(f"Found {len(chapters)} chapters:")
        for chapter in chapters:
            start_time = chapter['start_time'] # Time in seconds
            title = chapter['title']
            
            # Convert seconds to MM:SS format
            minutes = int(start_time // 60)
            seconds = int(start_time % 60)
            
            print(f"[{minutes:02d}:{seconds:02d}] {title}")
    else:
        print("This video does not have chapter markers.")

{'id': 'rV3HJ4LEZ7k', 'title': 'Complete Agentic AI Course In 10 Hours- Langchain, Langgraph, RAG,Vectorless RAG, Guardrails,Evals', 'formats': [{'format_id': 'sb3', 'format_note': 'storyboard', 'ext': 'mhtml', 'protocol': 'mhtml', 'acodec': 'none', 'vcodec': 'none', 'url': 'https://i.ytimg.com/sb/rV3HJ4LEZ7k/storyboard3_L0/default.jpg?sqp=-oaymwENSDfyq4qpAwVwAcABBqLzl_8DBgj6x7vQBg==&sigh=rs$AOn4CLDPjQ57q9hCMuk1M7kS39jNKvoM4A', 'width': 48, 'height': 27, 'fps': 0.0024749412201460217, 'rows': 10, 'columns': 10, 'fragments': [{'url': 'https://i.ytimg.com/sb/rV3HJ4LEZ7k/storyboard3_L0/default.jpg?sqp=-oaymwENSDfyq4qpAwVwAcABBqLzl_8DBgj6x7vQBg==&sigh=rs$AOn4CLDPjQ57q9hCMuk1M7kS39jNKvoM4A', 'duration': 40405.0}], 'audio_ext': 'none', 'video_ext': 'none', 'vbr': 0, 'abr': 0, 'tbr': None, 'resolution': '48x27', 'aspect_ratio': 1.78, 'filesize_approx': None, 'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/

In [20]:
import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi

MAX_NO_CHAPTER_DURATION = 3600  # 1 hour in seconds


def fmt_time(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def get_chaptered_transcript(video_id):
    video_url = f"https://www.youtube.com/watch?v={video_id}"

    # 1. Fetch metadata (title, description, duration, chapters)
    ydl_opts = {"quiet": True, "skip_download": True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=False)

    title = info.get("title", "")
    description = info.get("description", "")
    duration = info.get("duration", 0) or 0
    chapters = info.get("chapters")

    # 2. No chapters and longer than 1 hour -> stop before fetching the transcript
    if not chapters and duration > MAX_NO_CHAPTER_DURATION:
        print(
            f"Cannot be converted to blog: video has no chapters and is "
            f"{fmt_time(duration)} long (over 1 hour)."
        )
        return None

    # 3. Fetch the transcript
    try:
        ytt = YouTubeTranscriptApi()
        transcript_snippets = ytt.fetch(video_id)
    except Exception as e:
        print(f"Error fetching transcript: {e}")
        return None

    # 4. Build chunks
    if not chapters:
        # No chapters, but 1 hour or shorter: treat the whole video as one chunk
        full_text = " ".join(s.text.replace("\n", " ") for s in transcript_snippets)
        last = transcript_snippets[-1]
        chunks = [{
            "chapter_title": "Full Video",
            "start_time": 0,
            "end_time": last.start + last.duration,
            "text": full_text,
        }]
    else:
        chunks = []
        for chapter in chapters:
            start_time = chapter["start_time"]
            end_time = chapter["end_time"]

            chapter_lines = [
                s.text.replace("\n", " ")
                for s in transcript_snippets
                if start_time <= s.start < end_time
            ]

            chunks.append({
                "chapter_title": chapter["title"],
                "start_time": start_time,
                "end_time": end_time,
                "text": " ".join(chapter_lines),
            })

    return {
        "title": title,
        "description": description,
        "duration": duration,
        "has_chapters": bool(chapters),
        "chunks": chunks,
    }


if __name__ == "__main__":
    video_id = "rV3HJ4LEZ7k"  # Replace with your target video ID

    result = get_chaptered_transcript(video_id)

    if result is not None:
        print(f"Title: {result['title']}")
        print(f"Duration: {fmt_time(result['duration'])}")
        print(f"Has chapters: {result['has_chapters']}")
        print(f"\n=== Description ===\n{result['description']}\n")

        for chunk in result["chunks"]:
            print(f"\n--- Chapter: {chunk['chapter_title']} ---")
            print(f"Time: {fmt_time(chunk['start_time'])} -> {fmt_time(chunk['end_time'])}")
            print(f"Content: {chunk['text'][:150]}...")

Title: Complete Agentic AI Course In 10 Hours- Langchain, Langgraph, RAG,Vectorless RAG, Guardrails,Evals
Duration: 11:13:25
Has chapters: True

=== Description ===
This is a complete course on learning Generative ai and agentic with Langchain and Langgraph. We have included all the topics from RAG, vectorless rag, Deep agents, Guardrails, LLM Evaluation and LLM Gateways Techniques
Github Links
Langchain : https://github.com/krishnaik06/Langchain-V1-Crash-Course
Langgraph: https://github.com/krishnaik06/Agentic-LanggraphCrash-course
RAG: https://github.com/krishnaik06/RAG-Tutorials
Vectorless RAG: https://github.com/krishnaik06/RAG-Tutorials/blob/main/PageIndex_Vectorless_RAG_CrashCourse%20(1).ipynb
Deep Agents: https://drive.google.com/file/d/1SVjvgqvKfF-FPAIqpEZdKMQLhe13NLFD/view
Guardrails : https://github.com/krishnaik06/Langchain-V1-Crash-Course/blob/main/updatedlangchain/langchain_guardrails_crash_course.ipynb
LLM Evals : https://github.com/krishnaik06/RAG-Tutorials/blob/main/1-r

In [16]:
chapter_data

"Error fetching transcript: type object 'YouTubeTranscriptApi' has no attribute 'get_transcript'"